# Lax-beam prototype: direct comparison with PRL Eq. (3)

This notebook constructs the magnetic field from a Lax-expanded vector potential and compares it directly with Eq. (3) of M. Jirka and S. V. Bulanov, *Physical Review Letters* **133**, 125001 (2024).

Paper: [Phys. Rev. Lett. 133, 125001](https://doi.org/10.1103/PhysRevLett.133.125001)

The calculation follows the deliberately small prototype:

$$
\mathbf A
\longrightarrow
\mathbf B=\nabla\times\mathbf A
\longrightarrow
\text{truncate at }O(\epsilon^N)
\longrightarrow
\text{fast numerical function}.
$$

All mathematics in this notebook uses the standard Jupyter delimiters `$ ... $` and `$$ ... $$`. This avoids the rendering problem that some notebook front ends have with `\$...\$` and `\$$...\$$`.

The notebook emphasizes:

1. the coordinate relabelling between the PRL and our implementation;
2. why $B_y$ becomes the azimuthal field $B_\theta$ on a selected radial cut;
3. exact symbolic agreement with the PRL at orders $1$, $3$, and $5$;
4. conversion of the symbolic result to a NumPy-compatible function.


## 1. Imports and generic core

The file `lax_beams.py` contains only generic operations:

- Cartesian curl with optional derivative scales;
- Cartesian divergence;
- truncation in a formal parameter $\epsilon$;
- construction of $\mathbf B=\nabla\times\mathbf A$;
- `sympy.lambdify` compilation.

It contains no Gaussian-specific physics.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

candidate_dirs = [Path.cwd(), Path("/mnt/data/lax_beams_prototype")]

for candidate in candidate_dirs:
    if (candidate / "lax_beams.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lax_beams import (
    magnetic_field,
    divergence,
    truncate,
    compile_vector_field,
)

sp.init_printing()

## 2. Coordinate conventions and the azimuthal component

The PRL propagates the beam along its $x_{\rm p}$ axis. Its transverse radius is

$$
r_{\rm p}=\sqrt{y_{\rm p}^2+z_{\rm p}^2}.
$$

Our implementation propagates the beam along $z$. Its transverse radius is

$$
r=\sqrt{x^2+y^2}.
$$

We use the relabelling

$$
x_{\rm p}\longleftrightarrow z,
\qquad
y_{\rm p}\longleftrightarrow x,
\qquad
z_{\rm p}\longleftrightarrow y.
$$

Thus the paper's focal plane $x_{\rm p}=0$ is our plane $z=0$.

For a beam propagating along our $z$ axis, an azimuthal field has Cartesian components

$$
B_x=-\frac{y}{r}B_\theta,
\qquad
B_y=\frac{x}{r}B_\theta,
\qquad
B_z=0.
$$

On the positive $x$ axis, where $y=0$ and $x>0$,

$$
r=x,
\qquad
\hat{\boldsymbol\theta}=\hat{\mathbf y},
\qquad
B_y=B_\theta.
$$

This is the radial cut used for the direct comparison. In the paper's original coordinates it corresponds to $x_{\rm p}=0$, $z_{\rm p}=0$, and $y_{\rm p}>0$.


## 3. Normalized coordinates and diffraction parameter

For the Gaussian benchmark we use

$$
X=\frac{x}{w_0},
\qquad
Y=\frac{y}{w_0},
\qquad
Z=\frac{z}{z_R},
$$

with

$$
z_R=\frac{k w_0^2}{2},
\qquad
\epsilon=\frac{2}{k w_0}=\frac{\lambda}{\pi w_0}.
$$

Derivatives with respect to normalized variables are not physical derivatives:

$$
\frac{\partial}{\partial x}
=
\frac{1}{w_0}\frac{\partial}{\partial X}
=
\frac{k\epsilon}{2}\frac{\partial}{\partial X},
$$

$$
\frac{\partial}{\partial y}
=
\frac{k\epsilon}{2}\frac{\partial}{\partial Y},
$$

$$
\frac{\partial}{\partial z}
=
\frac{1}{z_R}\frac{\partial}{\partial Z}
=
\frac{k\epsilon^2}{2}\frac{\partial}{\partial Z}.
$$

A transverse derivative contributes one explicit power of $\epsilon$. This is why an even-order expansion of $A_z$ produces odd orders in $B_\theta$.


In [ ]:
I = sp.I

X, Y, Z = sp.symbols("X Y Z", real=True)
R = sp.symbols("R", nonnegative=True, real=True)

eps, k, A0 = sp.symbols("eps k A0", positive=True, real=True)

derivative_scales = (
    k * eps / 2,
    k * eps / 2,
    k * eps**2 / 2,
)

derivative_scales

## 4. Lax-expanded vector potential

For the benchmark we use a purely longitudinal vector potential,

$$
\mathbf A = \hat{\mathbf z}\,A_0\,\Psi,
$$

where

$$
\Psi
=
\Psi_0
+
\epsilon^2\Psi_2
+
\epsilon^4\Psi_4.
$$

Define

$$
\rho^2=X^2+Y^2,
\qquad
f(Z)=\frac{i}{i+Z}.
$$

The Gaussian factor is

$$
e^{-f\rho^2}.
$$

The three terms below are the analytic Lax coefficients used in the benchmark.

In [ ]:
rho2 = X**2 + Y**2
f = I / (I + Z)
gaussian = sp.exp(-f * rho2)

psi0 = f * gaussian

psi2 = (
    sp.Rational(1, 2) * f**2
    - sp.Rational(1, 4) * rho2**2 * f**4
) * gaussian

psi4 = (
    sp.Rational(3, 8) * f**3
    - sp.Rational(3, 16) * rho2**2 * f**5
    - sp.Rational(1, 8) * rho2**3 * f**6
    + sp.Rational(1, 32) * rho2**4 * f**7
) * gaussian

Psi = psi0 + eps**2 * psi2 + eps**4 * psi4

A = (
    sp.Integer(0),
    sp.Integer(0),
    A0 * Psi,
)

A

Because $A_x=A_y=0$, the curl has a particularly simple cylindrical interpretation.

For an axisymmetric longitudinal potential $A_z(r,z)$,

$$
\mathbf B
=
\nabla\times(A_z\hat{\mathbf z})
=
-\frac{\partial A_z}{\partial r}\,\hat{\boldsymbol\theta}.
$$

We will **not** use that cylindrical formula in the implementation. Instead we let the generic Cartesian curl compute $(B_x,B_y,B_z)$, then extract $B_\theta$ on the positive $X$-axis. There,

$$
\hat{\boldsymbol\theta}=\hat{\mathbf y},
$$

so $B_\theta=B_y$. Rotational symmetry then identifies the general radial coefficient.

## 5. Construct $\mathbf B$ at different Lax orders

The core function first computes the physical curl and then truncates the **final magnetic field** at the requested power of $\epsilon$.

We compare orders

$$
N=1,\quad 3,\quad 5.
$$

In [ ]:
B1 = magnetic_field(
    A, (X, Y, Z), eps, order=1,
    derivative_scales=derivative_scales,
)

B3 = magnetic_field(
    A, (X, Y, Z), eps, order=3,
    derivative_scales=derivative_scales,
)

B5 = magnetic_field(
    A, (X, Y, Z), eps, order=5,
    derivative_scales=derivative_scales,
)

B1

At first order only the paraxial contribution survives. Higher-order corrections enter successively through $\epsilon^3$ and $\epsilon^5$.

In [ ]:
def btheta_on_x_axis(B):
    # Return B_theta/(k A0) on Y=0, X=R.
    return sp.simplify((B[1] / (k * A0)).subs({X: R, Y: 0}))

btheta_1 = btheta_on_x_axis(B1)
btheta_3 = btheta_on_x_axis(B3)
btheta_5 = btheta_on_x_axis(B5)

btheta_1

## 6. Eq. (3) from the PRL

The PRL writes the azimuthal magnetic field as

$$
B_\theta
=
E_0 e^{-r_{\rm p}^2/w^2}
\left[
\epsilon\rho C_2
+
\epsilon^3
\left(
\frac{\rho C_3}{2}
+
\frac{\rho^3 C_4}{2}
-
\frac{\rho^5 C_5}{4}
\right)
+
\epsilon^5
\left(
\frac{3\rho C_4}{8}
+
\frac{3\rho^3 C_5}{8}
+
\frac{3\rho^5 C_6}{16}
-
\frac{\rho^7 C_7}{4}
+
\frac{\rho^9 C_8}{32}
\right)
\right].
$$

Here

$$
C_n=
\left(\frac{w_0}{w}\right)^n
\cos\left(\phi+n\phi_G\right),
$$

$$
w=w_0\sqrt{1+\left(\frac{x_{\rm p}}{x_R}\right)^2},
\qquad
\phi_G=\arctan\left(\frac{x_{\rm p}}{x_R}\right),
\qquad
\rho=\frac{r_{\rm p}}{w_0}.
$$

Now take the focal radial cut

$$
x_{\rm p}=0,
\qquad
z_{\rm p}=0,
\qquad
y_{\rm p}>0,
$$

which is our cut

$$
z=0,
\qquad
y=0,
\qquad
x>0.
$$

Then $w=w_0$, $\phi_G=0$, and $\rho=x/w_0=X$. At the phase $\phi_0+\omega_0t=0$, all $C_n=1$, so Eq. (3) reduces to

$$
\frac{B_\theta^{\rm PRL}}{E_0}
=
e^{-\rho^2}
\left[
\epsilon\rho
+
\epsilon^3
\left(
\frac{\rho}{2}
+
\frac{\rho^3}{2}
-
\frac{\rho^5}{4}
\right)
+
\epsilon^5
\left(
\frac{3\rho}{8}
+
\frac{3\rho^3}{8}
+
\frac{3\rho^5}{16}
-
\frac{\rho^7}{4}
+
\frac{\rho^9}{32}
\right)
\right].
$$

We compare this dimensionless shape with $B_y/(kA_0)$. Thus $E_0$ is identified with the magnetic amplitude scale $kA_0$ used by the prototype. In SI units, the corresponding physical magnetic scale is $E_0/c$.


In [ ]:
# Direct focal-plane reduction of PRL Eq. (3).
btheta_prl_focus = sp.exp(-R**2) * (
    eps * R
    + eps**3 * (
        R / 2
        + R**3 / 2
        - R**5 / 4
    )
    + eps**5 * (
        3 * R / 8
        + 3 * R**3 / 8
        + 3 * R**5 / 16
        - R**7 / 4
        + R**9 / 32
    )
)

btheta_prl_focus


## 7. Exact symbolic comparison on the focal radial cut

The field constructed from the vector potential is already expressed on the positive $X$ axis by `btheta_1`, `btheta_3`, and `btheta_5`. We now additionally set $Z=0$ and compare with the reduced PRL equation.

For each order, SymPy evaluates

$$
\left.
\frac{B_y^{\rm curl}}{kA_0}
\right|_{Y=0,\,Z=0,\,X=R}
-
\frac{B_\theta^{\rm PRL}}{E_0}.
$$

The expected result is exactly zero.


In [ ]:
comparisons = {}

for order, ours in [
    (1, btheta_1),
    (3, btheta_3),
    (5, btheta_5),
]:
    ours_at_focus = sp.simplify(ours.subs(Z, 0))
    paper_at_order = truncate(btheta_prl_focus, eps, order)
    difference = sp.simplify(ours_at_focus - paper_at_order)
    comparisons[order] = difference
    print(f"order {order}: difference = {difference}")


All three residuals are zero. Therefore

$$
\left.
\frac{B_y^{\rm curl}}{kA_0}
\right|_{Y=0,\,Z=0,\,X=R}
=
\frac{B_\theta^{\rm PRL}}{E_0}
$$

through $O(\epsilon^5)$.

The azimuthal interpretation can also be checked directly: on this cut the other Cartesian components vanish.


### Cartesian component check

On $Y=0$, $Z=0$, and $X=R>0$, an azimuthal field must satisfy

$$
B_x=0,
\qquad
B_y=B_\theta,
\qquad
B_z=0.
$$


In [ ]:
focus_line_components = tuple(
    sp.simplify(component.subs({X: R, Y: 0, Z: 0}))
    for component in B5
)

print("Bx on the focal radial cut:")
display(focus_line_components[0])

print("By on the focal radial cut:")
display(focus_line_components[1])

print("Bz on the focal radial cut:")
display(focus_line_components[2])


## 8. Why the propagated complex expression becomes the PRL functions $C_n$

The focal-plane test is the simplest exact comparison, but the relation is not accidental. The complex Gaussian factor satisfies

$$
f(Z)=\frac{1}{1-iZ}
=\frac{1}{\sqrt{1+Z^2}}e^{i\phi_G},
\qquad
\phi_G=\arctan Z.
$$

It also satisfies

$$
e^{-f\rho^2}
=
e^{-\rho^2/(1+Z^2)}
e^{-i\rho^2Z/(1+Z^2)}.
$$

The first factor is $e^{-r^2/w^2}$ and the second is the wavefront-curvature phase. Therefore, with the carrier convention used in the PRL,

$$
\operatorname{Re}
\left[
e^{i(\phi_0+\omega_0t-kz)}
e^{-f\rho^2}f^n
\right]
=
e^{-r^2/w^2}
\left(\frac{w_0}{w}\right)^n
\cos(\phi+n\phi_G).
$$

The right-hand side is exactly $e^{-r^2/w^2}C_n$. Thus the powers $f^n$ in the complex Lax field map term by term to the $C_n$ factors printed in Eq. (3).


## 9. Inspect the individual coefficients

It is also useful to look directly at the coefficient multiplying each odd power of $\epsilon$.

We extract

$$
[\epsilon^1]B_\theta,\qquad
[\epsilon^3]B_\theta,\qquad
[\epsilon^5]B_\theta.
$$

In [ ]:
expanded_B = sp.expand(btheta_5)

coeff_1 = sp.simplify(expanded_B.coeff(eps, 1))
coeff_3 = sp.simplify(expanded_B.coeff(eps, 3))
coeff_5 = sp.simplify(expanded_B.coeff(eps, 5))

print("Coefficient of eps:")
display(coeff_1)

print("Coefficient of eps^3:")
display(coeff_3)

print("Coefficient of eps^5:")
display(coeff_5)

This makes the parity structure especially clear:

- $\Psi$ was expanded in even powers $1,\epsilon^2,\epsilon^4,\ldots$;
- a transverse derivative contributes one additional power of $\epsilon$;
- therefore $B_\theta$ appears as $\epsilon,\epsilon^3,\epsilon^5,\ldots$.

## 10. Check $\nabla\cdot\mathbf B=0$

Because $\mathbf B=\nabla\times\mathbf A$, its divergence should vanish identically.

With a truncated asymptotic series it is safest to compute the divergence and then truncate the residual consistently to the same requested order.

In [ ]:
div_B5 = divergence(B5, (X, Y, Z), derivative_scales)

div_B5_truncated = sp.simplify(
    truncate(div_B5, eps, order=5)
)

div_B5_truncated

The result should again be exactly zero through the retained order.

## 11. Numerical evaluation at focus

At focus, $Z=0$ and $f=1$, so the field is real at the selected phase. We first overlay the fifth-order result obtained from the curl with the direct focal reduction of PRL Eq. (3), and then compare successive Lax orders.


In [ ]:
b1_fn = sp.lambdify((R, Z, eps), btheta_1, modules="numpy")
b3_fn = sp.lambdify((R, Z, eps), btheta_3, modules="numpy")
b5_fn = sp.lambdify((R, Z, eps), btheta_5, modules="numpy")
prl_focus_fn = sp.lambdify((R, eps), btheta_prl_focus, modules="numpy")

R_values = np.linspace(0.0, 2.5, 500)
eps_value = 0.3

y1 = np.real(b1_fn(R_values, 0.0, eps_value))
y3 = np.real(b3_fn(R_values, 0.0, eps_value))
y5 = np.real(b5_fn(R_values, 0.0, eps_value))
y_prl = np.real(prl_focus_fn(R_values, eps_value))

plt.figure(figsize=(7, 4.5))
plt.plot(R_values, y5, label="curl of vector potential")
plt.plot(R_values, y_prl, "--", label="PRL Eq. (3)")
plt.xlabel(r"$R=x/w_0$")
plt.ylabel(r"normalized $B_\theta$")
plt.title(r"Direct focal-cut comparison, $O(\epsilon^5)$")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("maximum numerical difference =", np.max(np.abs(y5 - y_prl)))

plt.figure(figsize=(7, 4.5))
plt.plot(R_values, y1, label=r"$O(\epsilon)$")
plt.plot(R_values, y3, label=r"$O(\epsilon^3)$")
plt.plot(R_values, y5, label=r"$O(\epsilon^5)$")
plt.xlabel(r"$R=x/w_0$")
plt.ylabel(r"normalized $B_\theta$")
plt.title(r"Successive Lax orders at focus, $\epsilon=0.3$")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


For small $\epsilon$, the higher-order curves should remain close to the paraxial result. Their separation gives a direct picture of the importance of nonparaxial corrections.

## 12. Size of the nonparaxial correction

The next plot isolates the correction relative to the previous truncation:

$$
\Delta_3
=
B_\theta^{(3)}-B_\theta^{(1)},
\qquad
\Delta_5
=
B_\theta^{(5)}-B_\theta^{(3)}.
$$

This is often more informative than comparing the full curves.

In [ ]:
delta3 = y3 - y1
delta5 = y5 - y3

plt.figure(figsize=(7, 4.5))
plt.plot(R_values, delta3, label=r"$B_\theta^{(3)}-B_\theta^{(1)}$")
plt.plot(R_values, delta5, label=r"$B_\theta^{(5)}-B_\theta^{(3)}$")
plt.xlabel(r"$R=r/w_0$")
plt.ylabel("normalized correction")
plt.title(r"Successive nonparaxial corrections at focus, $\epsilon=0.3$")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 13. How the profile changes with $\epsilon$

Because the expansion is asymptotic in the diffraction parameter, stronger focusing increases the importance of the higher-order terms.

Below we keep $Z=0$ and compare the fifth-order result for several values of $\epsilon$.

In [ ]:
for eps_value in (0.1, 0.3, 0.5):
    values = np.real(b5_fn(R_values, 0.0, eps_value))
    plt.figure(figsize=(7, 4.5))
    plt.plot(R_values, values)
    plt.xlabel(r"$R=r/w_0$")
    plt.ylabel(r"$B_\theta/(kA_0)$")
    plt.title(fr"$O(\epsilon^5)$ profile at focus, $\epsilon={eps_value}$")
    plt.grid(alpha=0.25)
    plt.show()

These plots should not be interpreted as a proof that a truncated Lax series remains quantitatively reliable for arbitrarily large $\epsilon$. They are simply a convenient way to inspect how rapidly the corrections grow.

## 14. Compile the full Cartesian field

So far we extracted $B_\theta$ symbolically. The actual generic output is the full Cartesian vector field

$$
\mathbf B(X,Y,Z).
$$

We now turn the fifth-order symbolic field into a NumPy-compatible callable.

For demonstration we set $k=A_0=1$. In real use these can be left as explicit function arguments or substituted with physical values.

In [ ]:
B5_normalized = tuple(
    sp.simplify(component.subs({k: 1, A0: 1}))
    for component in B5
)

B5_fn = compile_vector_field(
    B5_normalized,
    args=(X, Y, Z, eps),
)

x_values = np.array([0.0, 0.25, 0.5, 0.75])
y_values = np.zeros_like(x_values)
z_values = np.zeros_like(x_values)

Bx_num, By_num, Bz_num = B5_fn(
    x_values,
    y_values,
    z_values,
    0.3,
)

print("Bx =", np.asarray(Bx_num))
print("By =", np.asarray(By_num))
print("Bz =", np.asarray(Bz_num))

On the positive $X$-axis the magnetic field is expected to be purely azimuthal, so here it points along $+\hat y$ or $-\hat y$ depending on the phase/sign convention. The numerical output therefore provides a simple sanity check of the Cartesian representation.

## 15. What this prototype has established

The implementation verifies the direct chain

$$
\Psi_0+\epsilon^2\Psi_2+\epsilon^4\Psi_4
\longrightarrow
\mathbf A
\longrightarrow
\nabla\times\mathbf A
\longrightarrow
B_y
\longrightarrow
B_\theta^{\rm PRL}.
$$

On the focal radial cut

$$
z=0,
\qquad
y=0,
\qquad
x>0,
$$

we have $r=x$ and $B_y=B_\theta$. The symbolic differences from PRL Eq. (3) are exactly zero at orders $1$, $3$, and $5$.

The prototype also confirms:

- the even/odd Lax-order bookkeeping after physical derivative scaling;
- $\nabla\cdot\mathbf B=0$ through the retained order;
- successful conversion to a vectorized NumPy callable.

The scalar potential, electric field, pulse envelope, focal-plane propagation, and automated generation of higher-order Lax coefficients remain intentionally outside this first prototype.


## 16. Minimal reusable pattern

For another already-known Lax-expanded vector potential, the generic workflow is only:

```python
B_expr = magnetic_field(
    A=(Ax, Ay, Az),
    coords=(q1, q2, q3),
    eps=eps,
    order=N,
    derivative_scales=(s1, s2, s3),
)

B_fn = compile_vector_field(
    B_expr,
    args=(q1, q2, q3, eps, ...),
)
```

If `q1,q2,q3` are ordinary physical Cartesian coordinates, simply omit `derivative_scales`; the default is `(1,1,1)`.

The beam-specific quantities such as $w_0$, $k_\perp$, mode indices, or Rayleigh range belong in the analytic definition of $\mathbf A$, not in the generic magnetic-field constructor.